# 03 - Modelling (Local Smoke Test)

This notebook is a local smoke test to trial the full training pipeline. Its purpose is to verify that data loading, log1p transformation, AutoGluon training, evaluation, and MLflow logging all work correctly before submitting to Azure ML.

This was designed to run on CPU with a short time limit specified, so the resulting metrics are not meaningful as the model did not have enough time to converge. Full training is handled in `04_azure_ml_job.ipynb`, which submits the same logic (`src/train.py`) to a GPU compute cluster on Azure ML with a multi-hour time limit per target.

The target variables are: `Dry_Clover_g`, `Dry_Dead_g`, `Dry_Green_g`, `GDM_g`

Our approach is: One `MultiModalPredictor` per target. Targets are log1p-transformed before training to address right skew and zero inflation identified in EDA. Predictions are reversed with `expm1` so RMSE and R² are reported in the original grams scale.

## 0. Imports and configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import mlflow
from autogluon.multimodal import MultiModalPredictor

c:\Users\Kurtis\Documents\Data Analysis\Projects\image2biomass\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
root_dir      = Path("..")
processed_dir = root_dir / "data" / "processed"
raw_dir       = root_dir / "data" / "raw"
models_dir    = root_dir / "models"
models_dir.mkdir(exist_ok=True)

# The four biomass targets to predict (grams).
# Dry_Total_g is excluded, the EDA found it is a linear sum of the first three and would cause data leakage.
TARGETS = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "GDM_g"]

VAL_SIZE    = 0.2   # 20% held out for validation
RANDOM_SEED = 42   

# Time limit per target for the LOCAL smoke test.
# This is intentionally short, just to confirm the pipeline runs.
# The full training run uses thousands of seconds per target and is submitted via 04_azure_ml_job.ipynb.
TIME_LIMIT_PER_TARGET = 120

## 1. Load and prepare data

Image paths are stored as relative strings (`train/<filename>.jpg`) in `df_model.csv`. AutoGluon requires absolute paths to locate images on disk, so we resolve them here against `data/raw/`. We also verify that all image files exist before training.

In [3]:
df = pd.read_csv(processed_dir / "df_model.csv")

# Resolve relative image paths to absolute
df["image_path"] = df["image_path"].apply(lambda x: str(raw_dir / x))

# Verify images exist
missing = [p for p in df["image_path"] if not Path(p).exists()]
print(f"Total samples : {len(df)}")
print(f"Missing images: {len(missing)}")
print(f"Example path  : {df['image_path'].iloc[0]}")
df[TARGETS].describe().round(2)


Total samples : 357
Missing images: 0
Example path  : ..\data\raw\train\ID1011485656.jpg


,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,GDM_g
count,357.00,357.00,357.00,357.00
mean,6.65,12.04,26.62,33.27
std,12.12,12.40,25.40,24.94
min,0.00,0.00,0.00,1.04
25%,0.00,3.20,8.80,16.03
50%,1.42,7.98,20.80,27.11
75%,7.24,17.64,35.08,43.68
max,71.79,83.84,157.98,157.98


## 2. Train / validation split

Simple 80/20 stratification-free split. With only 357 samples we keep the split fixed via `random_state` to ensure reproducible evaluation across runs. The same split is used for every target so results are directly comparable.

In [4]:
df_train, df_val = train_test_split(
    df,
    test_size = VAL_SIZE,
    random_state = RANDOM_SEED,
)

print(f"Train: {len(df_train)} | Val: {len(df_val)}")

Train: 285 | Val: 72


## 3. Log-transform targets

All four targets are right-skewed and zero-inflated, which was confirmed in EDA. Training a regression model on raw skewed targets causes the loss function to be dominated by rare high-value samples, which distorts learning.

`np.log1p(x)` computes `log(x + 1)`, which compresses the skew and is safe for zeros (`log1p(0) = 0`). At evaluation time, `np.expm1()` reverses this so all metrics are reported in the original grams scale. We keep separate `df_train_log` and `df_val_log` copies so the original DataFrames remain available for computing final metrics on the raw scale.

In [ ]:
# Apply log1p to targets to compress right skew and handle zero inflation
# log1p(x) = log(x + 1), safe for zeros. Reverse at inference with np.expm1()
df_train_log = df_train.copy()
df_val_log = df_val.copy()

for target in TARGETS:
    df_train_log[target] = np.log1p(df_train_log[target])
    df_val_log[target] = np.log1p(df_val_log[target])

print("Target distributions after log1p transform:")
df_train_log[TARGETS].describe().round(3)

## 4. Train

One `MultiModalPredictor` is trained per target. AutoGluon automatically detects the image column (by file extension), the categorical columns (e.g. `State`), and the binary species indicator columns, and fuses them using a `MultimodalFusionMLP`, a pretrained image backbone combined with an MLP for tabular features.

Each predictor is saved to its own subdirectory under `models/` so they can be reloaded later without retraining. The MLflow run is opened here and stays active across the training and evaluation cells, so there is no need to restart the kernel between them.

In [ ]:
mlflow.set_experiment("pasture-biomass")

# Run name identifies this as a local smoke test so it is clearly distinct
# from the full GPU runs logged from Azure ML in 04_azure_ml_job.ipynb
mlflow.start_run(run_name="autogluon_smoke_test_local_cpu")

mlflow.log_params({
    "model_type"              : "AutoGluon MultiModalPredictor",
    "val_size"                : VAL_SIZE,
    "random_seed"             : RANDOM_SEED,
    "time_limit_per_target_s" : TIME_LIMIT_PER_TARGET,
    "targets"                 : str(TARGETS),
    "n_train"                 : len(df_train),
    "n_val"                   : len(df_val),
    "target_transform"        : "log1p",
    "compute"                 : "local_cpu",   # CPU here, but using GPU on Azure
})

predictors = {}
for target in TARGETS:
    print(f"\n--- Training: {target} ---")
    predictor = MultiModalPredictor(
        label=target,                                    
        problem_type="regression",
        path=str(models_dir / f"autogluon_{target}"),  
        eval_metric="rmse",                              
    )
    predictor.fit(
        train_data=df_train_log,         
        time_limit=TIME_LIMIT_PER_TARGET,
    )
    predictors[target] = predictor

print("\nAll targets trained.")

## 5. Evaluate

Predict on the held-out validation set for each target. Because the predictors were trained on log1p-transformed targets, their raw output is in log space. We reverse this with `np.expm1()` before computing RMSE and R² so all metrics are in the original grams scale and are directly interpretable. Metrics are also logged to the open MLflow run before it is closed.

In [ ]:
results = {}
for target in TARGETS:
    y_true = df_val[target].values

    y_pred = np.expm1(predictors[target].predict(df_val).values)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    results[target] = {"RMSE": round(rmse, 4), "R2": round(r2, 4)}

    mlflow.log_metrics({
        f"{target}_rmse": rmse,
        f"{target}_r2"  : r2,
    })

# Close the MLflow run
mlflow.end_run()

results_df = pd.DataFrame(results).T
print(results_df.to_string())

## 6. Next steps - full training on Azure ML

The metrics above are from a CPU smoke test with a 120-second time limit per target and are not representative of model performance. With only ~2 epochs completed, the model has not had time to converge.

In our test here, the model was still improving at the time limit. Further gains are expected with longer training runs.